# 0. Введение
Сейчас мы находимся в jupyter-ноутбуке (или ipython-ноутбуке). Это удобная среда для написания кода, проведения экспериментов, изучения данных, построения визуализаций и других нужд, не связанных с написанием production-кода.

Ноутбук состоит из ячеек, каждая из которых может быть либо ячейкой с кодом, либо ячейкой с текстом размеченным и неразмеченным. Текст поддерживает markdown-разметку и формулы в Latex.

Для работы с содержимым ячейки используется режим редактирования (Edit mode, включается нажатием клавиши Enter после выбора ячейки), а для навигации между ячейками искользуется командный режим (Command mode, включается нажатием клавиши Esc). Тип ячейки можно задать в командном режиме либо с помощью горячих клавиш (y to code, m to markdown, r to edit raw text), либо в меню Cell -> Cell type.

После заполнения ячейки нужно нажать Shift + Enter, эта команда обработает содержимое ячейки: проинтерпретирует код или сверстает размеченный текст.

# 1. Устанавка библиотек
`! pip install` - это команда позволит нам установить нужные библиотеки.  
В данном случае, `ipython-sql`,  `prettytable` нужны чтобы легко могли писать `sql` запросы, без каких либо оберток

In [1]:
! pip install ipython-sql prettytable==0.7.2

# Включение магической функции `sql`

In [3]:
%load_ext sql
%reload_ext sql

# Подключение к БД
Будем работать с диалектом `SQLite`, `SQLite` -  компактная встраиваемая СУБД, для него не стоить отдельный сервер поднимать, поэтому к нему проще всего подключиться)  
Ниже в картинке указана база данных `Northwind`, представляет простую схему для управления клиентами малого бизнеса, заказами, запасами, закупками, поставщиками, доставкой и сотрудниками  
Что стоит обратить в схеме
* Название таблицы (`Orders`, `Customers`, и так далее)
* Ключ таблицы (`OrderId`, `CustomerId`), заметьте, в некоторых таблицах (`OrderDetails`), 2 основных ключа (`OrderId`, `ProductId`), это значит что запись уникальная по этим колонкам. Примере, в заказе `OrderId` может быть несколько продуктов `ProductId`
* Каждой колонке указан тип данных, пример `nvarchar(20)` означает, что колонка содержит строке не длиннее 20 символов
* Также есть тип колонки `Nullable`, он означает что в записи данная колонка может иметь значение или может не иметь 

Типичные диаграммы называют - ERD диаграммой, https://www.lucidchart.com/pages/ru/erd-diagram

![База данных](Northwind_ERD.png)

In [4]:
%sql sqlite:///northwind.db
%config SqlMagic.style = 'DEFAULT'

# Пример работы с данными
Хотим вывести 10 случайных заказов

In [5]:
%%sql
SELECT 
    * 
FROM customers 
LIMIT 5;

 * sqlite:///northwind.db
Done.


CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,Western Europe,12209,Germany,030-0074321,030-0076545
ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,Central America,05021,Mexico,(5) 555-4729,(5) 555-3745
ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,Central America,05023,Mexico,(5) 555-3932,None
AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,British Isles,WA1 1DP,UK,(171) 555-7788,(171) 555-6750
BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,Northern Europe,S-958 22,Sweden,0921-12 34 65,0921-12 34 67


# Полезные ссылки, которые помогут решить ДЗ
Тут все про синтаксис работы `SQLite` - https://www.sqlitetutorial.net/sqlite-functions/
Что понадобится в ДЗ?
* `SELECT`, `FROM` - база
* `LIMIT` - выводит определенное кол-во строк
* `DISTINCT` - выводит уникальные строки по полю
* `ORDER BY` - сортирует строки
* `WHERE` - фильтрация строк
* `GROUP BY`
* `AS` - нужно чтобы переименовать колонку/таблицу
* `MIN`, `MAX`, `SUM`, `AVG`, `COUNT` - группировки (мин, макс, сумма, среднее, кол-во)
* `ROUND(number, 2)` - округляет кол-во цифр после запятой 
* `DATE(date_column, 'start of month')` - приводит дату `2024-01-05` к первому дня месяца `2024-01-01`
* `DATE(date_column, 'start of year')` - приводит дату `2024-01-05` к первому дня года `2024-01-01` 
* `CAST(column AS INT)` - приводит колонку к какому то типу
* `INNER JOIN` `LEFT JOINT` - нужно чтобы соединить по какому то правилу две таблицы

# Задача 0
Нужно вывести топ 5 городов пользователей, которые совершили заказы в 2016 году

In [6]:
%%sql
-- здесь код
SELECT
    DATE(orders.orderDate, 'start of year') AS order_year, -- можно было без этого, но просто хотел показать как работает функция
    customers.city                          AS city,  -- города пользователей
    COUNT(*)                                AS cnt, -- кол-во заказов
    COUNT(DISTINCT orders.customerID)       AS distinct_customer -- кол-во уникальных пользователей
FROM orders
    INNER JOIN customers
        ON customers.customerId = orders.customerId
WHERE orderDate >= '2016-01-01' AND orderDate < '2017-01-01' -- фильтруем даты (2016 год), '2016-01-01' автоматически переведется в тип DATE
GROUP BY 1, 2-- это говорит о том что, нужно группировать по 1, 2 колонке
ORDER BY 3 DESC -- это говорит о том что, нужно сортировать по 3 колонке
LIMIT 7 -- выводим 7 записей

 * sqlite:///northwind.db
Done.


order_year,city,cnt,distinct_customer
2016-01-01,London,89,6
2016-01-01,México D.F.,86,5
2016-01-01,Sao Paulo,65,4
2016-01-01,Madrid,56,3
2016-01-01,Rio de Janeiro,53,3
2016-01-01,Buenos Aires,42,3
2016-01-01,Portland,41,2


# Каждое задание имеет вес 1 балл

# Задание 1
* Выведите столбцы (`EmployeeID`, `CustomerID`, `RequiredDate`, `ShipCountry`) из таблицы `orders` (именно в таком порядке)
* Переименуйте поле `CustomerID` в `client_id`
* Отсортируйте по ключу `RequiredDate` в порядке убывания
* Ограничьтесь результатам 8 строками

Ответ:
<img src='result_img_autumn_2025/aut_25_task1.png' alt='Описание' width='300' height='200'>


In [12]:
%%sql
-- ваш код тут (это просто комментарий)



 * sqlite:///northwind.db
Done.


[]

# Задание 2
Выведите столбцы `SupplierID`, `CompanyName` и `City` из таблицы `suppliers` для поставщиков, у которых регион (`Region`) `North America`. Отсортируйте результат по `City` в алфавитном порядке.

Ответ:  
<img src='result_img_autumn_2025/aut_25_task2.png' alt='Описание' width='300' height='200'>

In [14]:
%%sql
-- ваш код тут


 * sqlite:///northwind.db
Done.


[]

# Задание 3
Выберите все столбцы из таблицы `suppliers`, отобразив только те у которых заполнен номер факса `Fax`, но не заполнена домашняя страница 
`HomePage` и отсортируйте товары по номеру телефона `Phone` по убыванию. Отобразите только первые 3 записи

Ответ:  
<img src='result_img_autumn_2025/aut_25_task3.png' alt='Описание' width='800' height='200'>

In [16]:
%%sql
-- ваш код тут



 * sqlite:///northwind.db
Done.


[]

# Задание 4
Подсчитайте, сколько клиентов в каждом городе `City` в таблице `customers`. Не учитывайте клиентов, у которых нет данных про их город.
В результате отобразите `City` и количество клиентов в столбце с алиасом (`AS`) `clientsCnt`, 
обязательно отсортируйте по `clientsCnt` в порядке убывания и отобразите первые 12 записей.

Ответ:  
<img src='result_img_autumn_2025/aut_25_task4.png' alt='Описание' width='200' height='300'>

In [22]:
%%sql
-- ваш код тут


 * sqlite:///northwind.db
Done.


[]

# Задание 5
Для каждого поставщика `SupplierID` из таблицы `Products` найдите минимальное, максимальное и среднее число товаров в продаже (`UnitsInStock`). 
Выведите `SupplierID` и столбцы с алиасами `minPrice`, `maxPrice`, `avgPrice`. 
Отсортируйте по полю `SupplierID` в порядке убывания, также округлите числа до `2` знаков после запятой, вам поможет функция `ROUND`. Выведите только 10 записей.

Ответ:  
<img src='result_img_autumn_2025/aut_25_task5.png' alt='Описание' width='400' height='300'>

In [24]:
%%sql
-- ваш код тут


 * sqlite:///northwind.db
Done.


[]

# Задание 6
* Сгруппируйте заказы (Orders) по первому дню месяца, переименуйте расчет в `monthStart` 
* Подсчитайте среднюю стоимость отгрузки `Freight` товара  в `avg_cost_per_order`, округлив её до двух знаков после запятой
* Отсортируйте по `monthStart`
* Выведите данные только за 2021 год


Ответ:  
<img src='result_img_autumn_2025/aut_25_task6.png' alt='Описание' width='300' height='500'>

In [26]:
%%sql
-- ваш код тут


 * sqlite:///northwind.db
Done.


[]

# Задание 7
* Из таблицы `orders` возьмите дату брони `RequiredDate` и округлите её до начала года, назвав стоблец `yearStart`
* В разрезе каждого года найдите первый и последний по алфавиту пункт отгрузки `ShipName`, а также дату первой и последней отгрузки `ShippedDate`
* Выведите столбцы `yearStart`, `firstShipName`, `lastShipName`, `firstShippedDate`, `lastShippedDate`, сортируя по `yearStart`

Ответ:  
<img src='result_img_autumn_2025/aut_25_task7.png' alt='Описание' width='300' height='500'>

In [28]:
%%sql
-- ваш код тут


 * sqlite:///northwind.db
Done.


[]

# Задание 8
Выведите страну отгрузки `ShipCountry`, максимальную цену отгрузки (`MaxFreight`) и максимальную цену отгрузки, приведённую к целому числу (нужно к результату применить функцию `CAST`). Приведенный `MaxFreight` выводите как поле `MaxFreightInt`. Отсортируйте по `MaxFreightInt`.

Ответ:  
<img src='result_img_autumn_2025/aut_25_task8.png' alt='Описание' width='600' height='500'>

In [30]:
%%sql
-- ваш код тут


 * sqlite:///northwind.db
Done.


[]

# Задание 9
Объедините таблицы `Orders` (псевдоним o, используйте `AS`) и `Customers` (псевдоним c) по полю `CustomerID`, чтобы вывести `OrderID`, `CompanyName` и `OrderDate`, `CustomerId`. Отсортируйте по `CustomerID`. Ограничьте вывод 10 строками.

Ответ:  
<img src='result_img_autumn_2025/aut_25_task9.png' alt='Описание' width='600' height='600'>

In [15]:
%%sql
-- ваш код тут



 * sqlite:///northwind.db
Done.


[]

# Задание 10
Объедините таблицы `Customers` (псевдоним `c`) и `Orders` (псевдоним `o`), чтобы для каждого города клиентов:  
* Подсчитать количество уникальных клиентов.
* Подсчитать общее число заказов, сделанных клиентами из этого города.
* Выведите столбцы:
    * `City` – город клиента
    * `CustomersCount` – количество уникальных клиентов в этом городе
    * `OrdersCount` – общее количество заказов.
* Отсортируйте результат по убыванию количества заказов и ограничьте вывод 10 строками.

Ответ:  
<img src='result_img_autumn_2025/aut_25_task10.png' alt='Описание' width='600' height='600'>


In [16]:
%%sql
-- ваш код тут



 * sqlite:///northwind.db
Done.


[]

# Placeholder для мемной картинки